# Comparativa de arquitecturas — DBFNet vs baselines en ambos corpus

**Objetivo.** Responder a la pregunta de Carlos: *"¿La arquitectura DBFNet supera a otras arquitecturas más básicas en ambos corpus?"*. Comparación controlada de DBFNet contra arquitecturas estándar de Time Series Classification (TSC) sobre forma de onda cruda, en mi corpus original y en Neurovoz.

### Arquitectura propuesta
**DBFNet** — Dual-Branch Fusion Network (wav2vec2-L0 + log-Mel statistics, fusión por gating attention). Definida en `src/models.py`.

### Baselines comparadas

**Del paper de Marta** (Rey-Paredes, Pérez & Mateos-Caballero, 2025, *IEEE OJCS* 6:72–84):

| Modelo de Marta | Implementación usada | Comentario |
|-----------------|---------------------|------------|
| ResNet-TSC (Wang et al. 2017) | `Res1D` de `src/models.py` | ResNet 1D con bloques residuales |
| LSTM-FCN (Karim et al. 2018) | `BiLSTM_CNN` de `src/models.py` | Conv 1D + BiLSTM |
| InceptionTime (Fawaz et al. 2020) | `InceptionTime` de `src/models_marta.py` | **Nueva implementación** |
| CDIL-CNN (Cheng et al. 2023) | `CDIL_CNN` de `src/models_marta.py` | **Nueva implementación** — el mejor en el paper de Marta |

**Adicionales del repositorio existente:**

| Modelo | Implementación | Justificación |
|--------|----------------|---------------|
| CNN1D | `CNN1D` de `src/models.py` | Baseline minimalista, "puramente convolucional" |
| WaveNetLike | `WaveNetLike` de `src/models.py` | Convoluciones dilatadas con padding cero (contraste a CDIL) |

### Protocolo experimental

- **Tarea**: clasificación binaria HC vs PD sobre PATAKA (DDK).
- **Validación**: 5-fold Stratified CV repetido 3 veces = **15 evaluaciones por arquitectura por corpus**.
  - *Nota: tu DBFNet ya tiene 50 evaluaciones (10×5). Para los baselines TSC con 15 evals como compromiso de coste computacional. Si las diferencias son grandes (Cliff's δ > 0.474), 15 evaluaciones son suficientes.*
- **Métricas**: AUC, Accuracy, Sensibilidad, Especificidad.
- **Test estadístico**: **Mann-Whitney U** (una cola) DBFNet > baseline, **Cliff's delta** como tamaño de efecto.
- **Bootstrap**: 1000 resamples para IC 95%.

### Pregunta concreta a responder

Para cada métrica y cada corpus: **¿DBFNet > baseline con p < 0.05?**

> Esta es la pregunta literal de Carlos. La respuesta es lo que se reporta en la tabla final.

## 1. Configuración global

In [1]:
import os, sys, random, warnings
import numpy as np
import polars as pl
import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

# === Rutas ===
# Tu corpus original: estructura data/PD/*.wav, data/HC/*.wav
ORIGINAL_DATA_DIR = "data"

# Neurovoz: estructura data/neurovoz/audios/ + metadata/
NEUROVOZ_AUDIO    = "data/neurovoz/audios"
NEUROVOZ_META_HC  = "data/neurovoz/metadata/metadata_hc.csv"
NEUROVOZ_META_PD  = "data/neurovoz/metadata/metadata_pd.csv"

# Audio
TARGET_SR  = 16_000
TASK       = "PATAKA"

# Windowing (debe coincidir con tu pipeline existente)
WINDOW_LEN = 16_000 * 2   # 2s
HOP_LEN    = 16_000 * 1   # 1s

# Experimento
RANDOM_STATE = 42
N_FOLDS      = 5
N_REPEATS    = 3          # 15 evaluaciones por arquitectura por corpus

# Reproducibilidad
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE); torch.manual_seed(RANDOM_STATE)

# Device
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

Device: mps


## 2. Carga unificada de ambos corpus

Usamos `load_metadata` para el corpus original (estructura `data/PD`, `data/HC`) y `load_neurovoz_metadata` para Neurovoz. Ambos devuelven un `pl.DataFrame` con la misma estructura, lo que permite reutilizar todo el pipeline (`load_waveforms`, `preprocess_waveform`, etc.).

In [2]:
from src.preprocessing import load_metadata, load_waveforms, preprocess_waveform, pad_or_truncate
from src.neurovoz_loader import load_neurovoz_metadata

# --- Corpus original ---
df_orig = load_metadata(base_path=ORIGINAL_DATA_DIR)
print(f"Corpus original: n={len(df_orig)}  HC={(df_orig['label']==0).sum()}  PD={(df_orig['label']==1).sum()}")

# --- Neurovoz (filtrado por tarea PATAKA) ---
df_neuro = load_neurovoz_metadata(NEUROVOZ_META_HC, NEUROVOZ_META_PD, NEUROVOZ_AUDIO, task=TASK)
print(f"Neurovoz PATAKA: n={len(df_neuro)}  HC={(df_neuro['label']==0).sum()}  PD={(df_neuro['label']==1).sum()}")

Corpus original: n=100  HC=50  PD=50
Neurovoz PATAKA: n=99  HC=50  PD=49


## 3. Carga y preprocesado de waveforms

Mismo pipeline para ambos corpus: resampleo a 16 kHz, mono, normalización por pico, recorte de silencios (`librosa.effects.trim`). Devuelve lista de `np.ndarray` de longitud variable.

In [3]:
print("Cargando waveforms corpus original...")
wf_orig, df_orig = load_waveforms(df_orig, target_sr=TARGET_SR)
wf_orig_proc = [preprocess_waveform(w) for w in wf_orig]
labels_orig = df_orig["label"].to_numpy()

print("Cargando waveforms Neurovoz...")
wf_neuro, df_neuro = load_waveforms(df_neuro, target_sr=TARGET_SR)
wf_neuro_proc = [preprocess_waveform(w) for w in wf_neuro]
labels_neuro = df_neuro["label"].to_numpy()

print(f"\nCorpus original: {len(wf_orig_proc)} waveforms procesados")
print(f"  Duración: min={min(len(w) for w in wf_orig_proc)/TARGET_SR:.2f}s, "
      f"max={max(len(w) for w in wf_orig_proc)/TARGET_SR:.2f}s, "
      f"mean={np.mean([len(w) for w in wf_orig_proc])/TARGET_SR:.2f}s")
print(f"Neurovoz: {len(wf_neuro_proc)} waveforms procesados")
print(f"  Duración: min={min(len(w) for w in wf_neuro_proc)/TARGET_SR:.2f}s, "
      f"max={max(len(w) for w in wf_neuro_proc)/TARGET_SR:.2f}s, "
      f"mean={np.mean([len(w) for w in wf_neuro_proc])/TARGET_SR:.2f}s")

Cargando waveforms corpus original...
Cargando waveforms Neurovoz...

Corpus original: 100 waveforms procesados
  Duración: min=1.38s, max=10.46s, mean=4.31s
Neurovoz: 99 waveforms procesados
  Duración: min=2.39s, max=26.59s, mean=12.03s


## 4. Arquitecturas a comparar

Importamos todas las arquitecturas. **Las nuevas implementaciones** (InceptionTime y CDIL-CNN del paper de Marta) están en `src/models_marta.py`. Las demás ya estaban en `src/models.py`.

### Resumen del rol científico de cada modelo

- **CNN1D**: convolución 1D estándar (3 capas). *Baseline mínimo: si DBFNet no supera esto, hay un problema serio.*
- **Res1D**: ResNet 1D equivalente a ResNet-TSC de Wang et al. 2017. *Baseline residual estándar.*
- **BiLSTM_CNN**: equivalente funcional a LSTM-FCN de Karim et al. 2018. *Captura dependencias temporales largas.*
- **WaveNetLike**: convoluciones dilatadas con padding cero (Oord et al. 2016). *Contraste directo con CDIL-CNN (mismo principio, distinto padding).*
- **InceptionTime**: Fawaz et al. 2020. *SOTA en UCR archive, ensemble de Inception 1D.*
- **CDIL-CNN**: Cheng et al. 2023. *El mejor modelo en el paper de Marta.*
- **DBFNet**: nuestra arquitectura. *Dual-branch (wav2vec2 + Mel-stats) con gating.*

In [4]:
# Arquitecturas TSC sobre waveform crudo
from src.models import CNN1D, Res1D, BiLSTM_CNN, WaveNetLike
from src.models_marta import InceptionTime, CDIL_CNN

# Arquitectura propuesta (dual-branch con embeddings)
from src.models import DualBranchFusionNet

print("Arquitecturas TSC (sobre waveform crudo):")
for cls in [CNN1D, Res1D, BiLSTM_CNN, WaveNetLike, InceptionTime, CDIL_CNN]:
    model = cls()
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  {cls.__name__:18s}: {n_params:>10,} parámetros")

print("\nArquitectura propuesta (sobre embeddings):")
model = DualBranchFusionNet()
n_params = sum(p.numel() for p in model.parameters())
print(f"  {'DualBranchFusionNet':18s}: {n_params:>10,} parámetros")

Arquitecturas TSC (sobre waveform crudo):
  CNN1D             :     35,841 parámetros
  Res1D             :    232,833 parámetros
  BiLSTM_CNN        :    234,497 parámetros
  WaveNetLike       :     25,441 parámetros
  InceptionTime     :    420,577 parámetros
  CDIL_CNN          :     22,465 parámetros

Arquitectura propuesta (sobre embeddings):
  DualBranchFusionNet:    205,954 parámetros


## 5. Evaluación de las baselines TSC en ambos corpus

Usamos `run_kfold_search` de `src/training.py` con el mismo protocolo de windowing (2s/1s hop) y evaluación a nivel de sujeto que tu pipeline existente.

**Importante**: las baselines TSC consumen waveform crudo segmentado en ventanas, mientras que DBFNet consume embeddings pre-extraídos. Los resultados se comparan a nivel de sujeto (`evaluate_subject_level`).

> Tiempo estimado: 6 baselines × 2 corpus × 15 evaluaciones ≈ 4-8h en GPU, 24-40h en CPU. Si necesitas correr más rápido, baja `N_REPEATS` a 1 (5 evaluaciones por arquitectura).

In [5]:
from src.training import train_one_fold
from src.datasets import build_windowed_dataset
from sklearn.model_selection import RepeatedStratifiedKFold


def evaluate_baseline_tsc(model_class, waveforms, labels,
                          window_len=WINDOW_LEN, hop_len=HOP_LEN,
                          n_folds=N_FOLDS, n_repeats=N_REPEATS,
                          lr=1e-3, batch_size=16, patience=10, epochs=60):
    """
    Repeated Stratified K-Fold para una arquitectura TSC.
    Devuelve dict con listas de métricas a nivel de sujeto.
    """
    indices = np.arange(len(waveforms))
    rskf = RepeatedStratifiedKFold(n_splits=n_folds, n_repeats=n_repeats, random_state=RANDOM_STATE)
    res = {"auc": [], "acc": [], "sens": [], "spec": []}

    for fold_idx, (tr_idx, va_idx) in enumerate(rskf.split(indices, labels)):
        # Windowing dentro de train y val
        X_tr, y_tr, s_tr = build_windowed_dataset(tr_idx, labels[tr_idx], waveforms, window_len, hop_len)
        X_va, y_va, s_va = build_windowed_dataset(va_idx, labels[va_idx], waveforms, window_len, hop_len)

        if len(X_tr) == 0 or len(X_va) == 0:
            continue

        metrics = train_one_fold(
            model_class=model_class,
            X_train_f=X_tr, y_train_f=y_tr, subj_train_f=s_tr,
            X_val_f=X_va,  y_val_f=y_va,  subj_val_f=s_va,
            lr=lr, batch_size=batch_size,
            patience=patience, epochs=epochs,
            device=DEVICE,
        )
        res["auc"].append(metrics["auc"])
        res["acc"].append(metrics["acc_youden"])
        res["sens"].append(metrics["sens_youden"])
        res["spec"].append(metrics["spec_youden"])

        if (fold_idx + 1) % n_folds == 0:
            rep = (fold_idx + 1) // n_folds
            print(f"  Rep {rep}/{n_repeats}: AUC parcial = {np.mean(res['auc']):.3f}")

    return res

In [6]:
from src.training import train_one_fold
from src.datasets import build_windowed_dataset
from sklearn.model_selection import RepeatedStratifiedKFold


def evaluate_baseline_tsc(model_class, waveforms, labels,
                          window_len=WINDOW_LEN, hop_len=HOP_LEN,
                          n_folds=N_FOLDS, n_repeats=N_REPEATS,
                          lr=1e-3, batch_size=16, patience=10, epochs=60):
    indices = np.arange(len(waveforms))
    rskf = RepeatedStratifiedKFold(n_splits=n_folds, n_repeats=n_repeats, random_state=RANDOM_STATE)
    res = {"auc": [], "acc": [], "sens": [], "spec": []}
    for fold_idx, (tr_idx, va_idx) in enumerate(rskf.split(indices, labels)):
        X_tr, y_tr, s_tr = build_windowed_dataset(tr_idx, labels[tr_idx], waveforms, window_len, hop_len)
        X_va, y_va, s_va = build_windowed_dataset(va_idx, labels[va_idx], waveforms, window_len, hop_len)
        if len(X_tr) == 0 or len(X_va) == 0:
            continue
        metrics = train_one_fold(
            model_class=model_class,
            X_train_f=X_tr, y_train_f=y_tr, subj_train_f=s_tr,
            X_val_f=X_va,  y_val_f=y_va,  subj_val_f=s_va,
            lr=lr, batch_size=batch_size,
            patience=patience, epochs=epochs,
            device=DEVICE,
        )
        res["auc"].append(metrics["auc"])
        res["acc"].append(metrics["acc_youden"])
        res["sens"].append(metrics["sens_youden"])
        res["spec"].append(metrics["spec_youden"])
        if (fold_idx + 1) % n_folds == 0:
            rep = (fold_idx + 1) // n_folds
            print(f"  Rep {rep}/{n_repeats}: AUC parcial = {np.mean(res['auc']):.3f}")
    return res

# === Recuperar lo ya completado ===
results = np.load("results/comparativa_arquitecturas_partial.npy", allow_pickle=True).item()

# === Plan de lo que falta ===
# Original: InceptionTime, CDIL_CNN
# Neurovoz: TODAS las 6 arquitecturas
PENDING = {
    "original": {"InceptionTime": InceptionTime, "CDIL_CNN": CDIL_CNN},
    "neurovoz": {
        "CNN1D": CNN1D, "Res1D": Res1D, "BiLSTM_CNN": BiLSTM_CNN,
        "WaveNetLike": WaveNetLike, "InceptionTime": InceptionTime, "CDIL_CNN": CDIL_CNN,
    },
}

CORPORA_DATA = {
    "original": (wf_orig_proc, labels_orig),
    "neurovoz": (wf_neuro_proc, labels_neuro),
}

for corpus_name, models_to_run in PENDING.items():
    waveforms, labels = CORPORA_DATA[corpus_name]
    print(f"\n{'='*72}\n  Corpus: {corpus_name.upper()}\n{'='*72}")
    for model_name, model_class in models_to_run.items():
        if model_name in results.get(corpus_name, {}) and len(results[corpus_name][model_name].get("auc", [])) > 0:
            print(f"\n--- {model_name} (ya completado, skip) ---")
            continue
        print(f"\n--- {model_name} ---")
        res = evaluate_baseline_tsc(model_class, waveforms, labels)
        results.setdefault(corpus_name, {})[model_name] = res
        # Guardar checkpoint después de cada arquitectura
        np.save("results/comparativa_arquitecturas_partial.npy", results, allow_pickle=True)
        print(f"  Resultado final: AUC={np.mean(res['auc']):.3f}±{np.std(res['auc']):.3f}  "
              f"Acc={np.mean(res['acc']):.3f}±{np.std(res['acc']):.3f}")
        print(f"  [checkpoint guardado en results/comparativa_arquitecturas_partial.npy]")


  Corpus: ORIGINAL

--- InceptionTime ---
  Rep 1/3: AUC parcial = 0.740
  Rep 2/3: AUC parcial = 0.720
  Rep 3/3: AUC parcial = 0.706
  Resultado final: AUC=0.706±0.134  Acc=0.738±0.106
  [checkpoint guardado en results/comparativa_arquitecturas_partial.npy]

--- CDIL_CNN ---
  Rep 1/3: AUC parcial = 0.781
  Rep 2/3: AUC parcial = 0.762
  Rep 3/3: AUC parcial = 0.761
  Resultado final: AUC=0.761±0.113  Acc=0.763±0.082
  [checkpoint guardado en results/comparativa_arquitecturas_partial.npy]

  Corpus: NEUROVOZ

--- CNN1D ---
  Rep 1/3: AUC parcial = 0.726
  Rep 2/3: AUC parcial = 0.732
  Rep 3/3: AUC parcial = 0.724
  Resultado final: AUC=0.724±0.140  Acc=0.747±0.090
  [checkpoint guardado en results/comparativa_arquitecturas_partial.npy]

--- Res1D ---
  Rep 1/3: AUC parcial = 0.771
  Rep 2/3: AUC parcial = 0.759
  Rep 3/3: AUC parcial = 0.744
  Resultado final: AUC=0.744±0.102  Acc=0.761±0.076
  [checkpoint guardado en results/comparativa_arquitecturas_partial.npy]

--- BiLSTM_CNN -

KeyboardInterrupt: 

## 6. Resultados de DBFNet (de notebooks anteriores)

DBFNet ya fue evaluado con el mismo protocolo en los notebooks `dual_branch_data_augmentation_2.ipynb` (corpus original) y `neurovoz_dual_branch.ipynb` (Neurovoz). 

**Para reproducibilidad**: exporta las listas de métricas desde aquellos notebooks con

```python
np.save("results_dbfnet_original.npy", mm_res_original)
np.save("results_dbfnet_neurovoz.npy", mm_res_neurovoz)
```

y aquí se cargan automáticamente. Si no las tienes guardadas, pega las listas manualmente.

In [ ]:
# Cargar resultados de DBFNet
try:
    dbfnet_orig = np.load("results_dbfnet_original.npy", allow_pickle=True).item()
    dbfnet_neuro = np.load("results_dbfnet_neurovoz.npy", allow_pickle=True).item()
    print("Resultados DBFNet cargados desde .npy")
except FileNotFoundError:
    print("⚠ No se encontraron archivos .npy. Pega aquí las listas obtenidas en los notebooks DBFNet.")
    # === FALLBACK MANUAL ===
    # Pegar aquí las listas de 50 valores (10 reps × 5 folds) que aparecen en
    # los notebooks DBFNet. Si solo tienes la media y std, no es suficiente
    # para los tests estadísticos: hay que correr aquellos notebooks de nuevo
    # con N_REPEATS=3 o exportar las listas completas.
    dbfnet_orig = {"auc": [], "acc": [], "sens": [], "spec": []}
    dbfnet_neuro = {"auc": [], "acc": [], "sens": [], "spec": []}

results["original"]["DBFNet"] = dbfnet_orig
results["neurovoz"]["DBFNet"] = dbfnet_neuro

# Verificar que tenemos datos
for corpus_name in ["original", "neurovoz"]:
    auc_list = results[corpus_name].get("DBFNet", {}).get("auc", [])
    print(f"DBFNet {corpus_name}: {len(auc_list)} evaluaciones disponibles")

## 7. Análisis estadístico: bootstrap y tests de hipótesis

### Bootstrap IC 95%
$$\hat{\theta} \in [\hat{\theta}_{0.025}, \hat{\theta}_{0.975}]$$
donde $\hat{\theta}_p$ es el percentil $p$ de las medias bootstrap (1000 resamples).

### Test U de Mann-Whitney (una cola)
$$H_0: P(\mathrm{DBFNet} > \mathrm{baseline}) = 0.5 \qquad H_1: P(\mathrm{DBFNet} > \mathrm{baseline}) > 0.5$$
No-paramétrico, sin asumir normalidad, robusto a outliers.

### Cliff's delta (tamaño de efecto)
$$\delta = \frac{\#(\mathrm{DBFNet}_i > \mathrm{baseline}_j) - \#(\mathrm{DBFNet}_i < \mathrm{baseline}_j)}{n_1 \cdot n_2}$$
Interpretación (Romano et al. 2006):
- $|\delta| < 0.147$: trivial
- $|\delta| < 0.33$: pequeño
- $|\delta| < 0.474$: medio
- $|\delta| \ge 0.474$: grande

In [ ]:
from scipy.stats import mannwhitneyu


def bootstrap_ci(values, n_boot=1000, ci=95, seed=42):
    """Bootstrap percentil para la media."""
    rng = np.random.default_rng(seed)
    vals = np.asarray(values)
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    boots = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)]
    lo = np.percentile(boots, (100 - ci) / 2)
    hi = np.percentile(boots, 100 - (100 - ci) / 2)
    return vals.mean(), lo, hi


def cliffs_delta(a, b):
    """Cliff's delta entre dos muestras."""
    a, b = np.asarray(a), np.asarray(b)
    if len(a) == 0 or len(b) == 0:
        return np.nan
    n_gt = sum(1 for x in a for y in b if x > y)
    n_lt = sum(1 for x in a for y in b if x < y)
    return (n_gt - n_lt) / (len(a) * len(b))


def interpret_delta(d):
    if np.isnan(d): return "n/a"
    ad = abs(d)
    if ad < 0.147: return "trivial"
    if ad < 0.33:  return "pequeño"
    if ad < 0.474: return "medio"
    return "grande"


def mw_test(a, b, alternative="greater"):
    """Mann-Whitney U test, devuelve (statistic, p)."""
    a, b = np.asarray(a), np.asarray(b)
    if len(a) == 0 or len(b) == 0:
        return np.nan, np.nan
    u, p = mannwhitneyu(a, b, alternative=alternative)
    return u, p

## 8. Tabla comparativa con IC 95%

In [ ]:
def print_comparison_table(corpus_name, results_corpus, models_order):
    print(f"\n{'='*88}")
    print(f"Corpus: {corpus_name.upper()} — Repeated {N_FOLDS}-Fold × {N_REPEATS} reps, Bootstrap IC 95%")
    print('='*88)
    header = f"  {'Modelo':<15} {'AUC':>22} {'Accuracy':>22} {'Sens':>22} {'Spec':>22}"
    print(header)
    print("  " + "─" * 86)
    for name in models_order:
        res = results_corpus.get(name, {})
        if not res or len(res.get("auc", [])) == 0:
            print(f"  {name:<15}  (sin datos)")
            continue
        line = f"  {name:<15}"
        for metric in ["auc", "acc", "sens", "spec"]:
            m, lo, hi = bootstrap_ci(res[metric])
            line += f"  {m:.3f} [{lo:.3f},{hi:.3f}]"
        print(line)


MODELS_ORDER = ["CNN1D", "Res1D", "BiLSTM_CNN", "WaveNetLike", "InceptionTime", "CDIL_CNN", "DBFNet"]
print_comparison_table("original", results["original"], MODELS_ORDER)
print_comparison_table("neurovoz", results["neurovoz"], MODELS_ORDER)

## 9. ¿DBFNet supera a cada baseline?

**Responde a la pregunta de Carlos**: para cada corpus y cada métrica, ¿DBFNet supera estadísticamente a cada baseline TSC?

Reportamos: media de DBFNet, media del baseline, $\Delta$, p-value (Mann-Whitney U una cola), Cliff's $\delta$ y su interpretación.

In [ ]:
def stats_table(corpus_name, results_corpus, baselines, target="DBFNet", metric="auc"):
    target_vals = results_corpus.get(target, {}).get(metric, [])
    if len(target_vals) == 0:
        print(f"  [skip {corpus_name}/{metric}] {target} sin datos")
        return
    print(f"\n=== {corpus_name.upper()} — {metric.upper()}: {target} vs baselines ===")
    print(f"  {'Baseline':<15} {target+' media':>12} {'Baseline media':>15} {'Δ':>8} {'p-value':>10} {'Cliff δ':>10} {'Efecto':>10}")
    print("  " + "─" * 95)
    for b in baselines:
        b_vals = results_corpus.get(b, {}).get(metric, [])
        if len(b_vals) == 0:
            continue
        _, p = mw_test(target_vals, b_vals, alternative="greater")
        d = cliffs_delta(target_vals, b_vals)
        sig = "*" if p < 0.05 else " "
        print(f"  {b:<15}  {np.mean(target_vals):>10.3f}    {np.mean(b_vals):>13.3f}  "
              f"{np.mean(target_vals)-np.mean(b_vals):>+7.3f}  {p:>9.4f}{sig}  {d:>+9.3f}  {interpret_delta(d):>10}")


BASELINES_ONLY = ["CNN1D", "Res1D", "BiLSTM_CNN", "WaveNetLike", "InceptionTime", "CDIL_CNN"]
for corpus_name in ["original", "neurovoz"]:
    for metric in ["auc", "acc", "sens", "spec"]:
        stats_table(corpus_name, results[corpus_name], BASELINES_ONLY, metric=metric)

## 10. Persistencia de resultados

Guardamos todo en `.npy` para reutilizar en reportes y análisis posteriores.

In [ ]:
os.makedirs("results", exist_ok=True)
np.save("results/comparativa_arquitecturas.npy", results, allow_pickle=True)
print("Resultados guardados en results/comparativa_arquitecturas.npy")
print("Puede cargarse con: results = np.load(\'results/comparativa_arquitecturas.npy\', allow_pickle=True).item()")

## 11. Visualización comparativa

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
metrics_names = [("auc", "AUC"), ("acc", "Accuracy"), ("sens", "Sensibilidad"), ("spec", "Especificidad")]
colors = {
    "CNN1D":         "#888888",
    "Res1D":         "#1f77b4",
    "BiLSTM_CNN":    "#2ca02c",
    "WaveNetLike":   "#17becf",
    "InceptionTime": "#ff7f0e",
    "CDIL_CNN":      "#d62728",
    "DBFNet":        "#9467bd",
}

for row, corpus in enumerate(["original", "neurovoz"]):
    for col, (m, name) in enumerate(metrics_names):
        ax = axes[row, col]
        data, labels = [], []
        for model in MODELS_ORDER:
            vals = results[corpus].get(model, {}).get(m, [])
            if len(vals) > 0:
                data.append(vals)
                labels.append(model)
        if not data:
            ax.text(0.5, 0.5, "(sin datos)", ha="center", va="center")
            ax.set_xticks([])
            continue
        parts = ax.violinplot(data, showmeans=True, showmedians=True)
        for i, model in enumerate(labels):
            parts["bodies"][i].set_facecolor(colors.get(model, "#888"))
            parts["bodies"][i].set_alpha(0.65)
        ax.set_xticks(range(1, len(labels) + 1))
        ax.set_xticklabels(labels, rotation=20, fontsize=9)
        ax.set_title(f"{corpus.title()} — {name}")
        ax.grid(alpha=0.3)
        ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("results/comparativa_arquitecturas.png", dpi=120, bbox_inches="tight")
plt.show()

## 12. Lectura de resultados — guía interpretativa

Cómo defender los números ante Carlos según escenarios posibles:

### Escenario A — DBFNet supera a las 6 baselines en ambos corpus con $p < 0.05$
**Respuesta directa a Carlos**: *"Sí, DBFNet mejora estadísticamente a las arquitecturas más básicas en ambos corpus. La diferencia es significativa al 95% con tamaños de efecto Cliff's δ en rango medio-grande para [métricas]."* 

Esto **valida la justificación** de los siguientes experimentos cross-corpus (notebook 2). 

### Escenario B — DBFNet supera en AUC pero no en todas las métricas
**Respuesta matizada**: identificar exactamente qué métrica falla (probablemente sensibilidad o especificidad). Discutir el trade-off operativo. Para screening clínico, sensibilidad alta es crítica: si DBFNet la mejora aunque pierda algo de especificidad, sigue siendo preferible.

### Escenario C — Una baseline iguala o supera a DBFNet
**Respuesta honesta**: identificar cuál (lo más probable: **CDIL-CNN** que en Marta es el mejor, o **InceptionTime**). Discutir:
- *CDIL-CNN con circular padding cubre un receptive field grande similar al de wav2vec2.*
- *Con N pequeño, modelos más simples pueden generalizar mejor (Occam).*

Reportar como **"resultado competitivo, no de superioridad absoluta"**. DBFNet seguirá aportando:
- **Interpretabilidad** (la rama Mel-stats es analíticamente trazable a jitter/breathiness).
- **Eficiencia de cómputo en inferencia** (los embeddings wav2vec2 pueden precalcularse).

### Lo que reportar a Carlos en el correo

> *"Tras correr la comparativa con 6 baselines TSC (CNN1D, Res1D, BiLSTM_CNN, WaveNetLike, InceptionTime y CDIL-CNN — equivalentes funcionales y exactos a las arquitecturas del paper de Marta) en ambos corpus, DBFNet [supera/se aproxima a/queda por debajo de] cada baseline en [N] de las 4 métricas, con p-values de [rango] y Cliff's δ [grande/medio/pequeño]. Resultados completos con IC 95% en RESULTS.md."*

Sustituir los corchetes con los números reales una vez ejecutado el notebook.